In [1]:
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, recall_score, f1_score, precision_score, \
    precision_recall_curve, roc_auc_score, average_precision_score
import tensorflow as tf
from sklearn.utils import resample
#from tensorflow.keras import layers, models      - if using tf 2.0 or higher, directly use tf.keras
import keras
from keras import layers, models
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


BASE_PATH = '1.0.3/training_data'
SAMPLE_RATE = 4000
N_MELS = 128
HOP_LENGTH = 256
N_FFT = 1024
DURATION = 5  # seconds

TARGET_SHAPE = (N_MELS, int(SAMPLE_RATE * DURATION / HOP_LENGTH) + 1)

N_FOLDS = 5
BATCH_SIZE = 32
NUM_EPOCHS = 100
LEARNING_RATE = 1e-4
MIN_SENSITIVITY = 0.8

In [2]:
from google.colab import drive
drive.mount('/content/drive')
DataPath = "drive/MyDrive/final project" # Change to your path
%cd $DataPath

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1XSXz2cLc1XyVo_YQgyusl-fliUqxOiQr/final project


In [3]:
cohort = pd.read_csv('cohort.csv')
print(f"Total patients: {len(cohort)}")
print(f"Murmur distribution:\n{cohort['Murmur'].value_counts()}")
print(f"Outcome distribution:\n{cohort['Outcome'].value_counts()}")

Total patients: 942
Murmur distribution:
Murmur
Absent     695
Present    179
Unknown     68
Name: count, dtype: int64
Outcome distribution:
Outcome
Normal      486
Abnormal    456
Name: count, dtype: int64


In [4]:
def extract_mel_spectrogram(wav_path, sr=SAMPLE_RATE, n_mels=N_MELS, duration=DURATION):
    """
    Convert PCG audio to 2D Mel spectrogram
    Captures frequency-domain features including high-frequency turbulence
    """
    try:
        signal, sr = librosa.load(wav_path, sr=sr, duration=duration)

        # Pad or truncate to fixed length
        target_length = sr * duration
        if len(signal) < target_length:
            signal = np.pad(signal, (0, target_length - len(signal)), mode='constant')
        else:
            signal = signal[:target_length]

        mel_spec = librosa.feature.melspectrogram(
            y=signal,
            sr=sr,
            n_mels=n_mels,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            fmin=20,      # Min frequency for heart sounds
            fmax=1000     # Max frequency for murmurs
        )

        # Convert to log scale (dB)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        return mel_spec_db

    except Exception as e:
        print(f"Error processing {wav_path}: {e}")
        return None

In [5]:
def process_all_recordings(cohort_df, base_path):
    """
    Process all patient recordings and extract Mel spectrograms
    """
    spectrograms = []
    labels_murmur = []
    labels_outcome = []
    patient_ids = []
    locations = []

    for idx, row in tqdm(cohort_df.iterrows(), total=len(cohort_df), desc="Processing audio"):
        patient_id = row['patient_id']

        try:
            recordings = eval(row['recordings'])
        except:
            continue

        for rec in recordings:
            wav_file = rec['wav_file']
            location = rec['location']
            wav_path = os.path.join(base_path, wav_file)

            if os.path.exists(wav_path):
                mel_spec = extract_mel_spectrogram(wav_path)

                if mel_spec is not None:
                    spectrograms.append(mel_spec)
                    labels_murmur.append(row['Murmur'])
                    labels_outcome.append(row['Outcome'])
                    patient_ids.append(patient_id)
                    locations.append(location)
            else:
                print(f"File not found: {wav_path}")

    return (np.array(spectrograms),
            np.array(labels_murmur),
            np.array(labels_outcome),
            np.array(patient_ids),
            np.array(locations))


print("Extracting Mel spectrograms...")
X, y_murmur, y_outcome, patient_ids, locations = process_all_recordings(cohort, BASE_PATH)

print(f"\nDataset shape: {X.shape}")
print(f"Unique patients: {len(np.unique(patient_ids))}")
print(f"Recording locations: {np.unique(locations)}")

Extracting Mel spectrograms...


Processing audio: 100%|██████████| 942/942 [18:56<00:00,  1.21s/it]


Dataset shape: (3163, 128, 79)
Unique patients: 942
Recording locations: ['AV' 'MV' 'PV' 'Phc' 'TV']


In [6]:
# Check for GPU
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

# If GPU is available, TensorFlow/Keras will use it automatically
# Optional: Limit GPU memory growth (prevents OOM errors)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Using GPU: {gpus[0].name}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found - using CPU")

Num GPUs Available: 1
Using GPU: /physical_device:GPU:0


In [7]:
import matplotlib
matplotlib.use('Agg')
def plot_sample_spectrograms(X, y, labels_to_show=['Present', 'Absent']):
    """
    Visualize Mel spectrograms for different murmur classes
    """
    fig, axes = plt.subplots(1, len(labels_to_show), figsize=(12, 4))

    for i, label in enumerate(labels_to_show):
        idx = np.where(y == label)[0]
        if len(idx) > 0:
            sample_idx = idx[0]
            librosa.display.specshow(
                X[sample_idx],
                sr=SAMPLE_RATE,
                hop_length=HOP_LENGTH,
                x_axis='time',
                y_axis='mel',
                ax=axes[i]
            )
            axes[i].set_title(f'Murmur: {label}')
            axes[i].set_xlabel('Time (s)')
            axes[i].set_ylabel('Frequency (Hz)')

    plt.tight_layout()
    plt.savefig('m2/mel_spectrogram_samples.png', dpi=150)
    plt.close(fig)

plot_sample_spectrograms(X, y_murmur)

In [8]:
mask = y_murmur != 'Unknown'
X_filtered = X[mask]
y_filtered = y_murmur[mask]
patient_ids_filtered = patient_ids[mask]
locations_filtered = locations[mask]

print(f"\nFiltered dataset (excluding Unknown): {X_filtered.shape[0]} samples")

y_encoded = (y_filtered == 'Present').astype(int)
print(f"\nEncoded labels - Present (1): {y_encoded.sum()}, Absent (0): {(y_encoded==0).sum()}")

X_filtered = X_filtered[..., np.newaxis]


X_mean = X_filtered.mean()
X_std = X_filtered.std()
X_normalized = (X_filtered - X_mean) / X_std


# patient-level train-test split to prevent data leakage
patient_df = pd.DataFrame({
    'patient_id': patient_ids_filtered,
    'label': y_encoded
}).drop_duplicates(subset='patient_id')

unique_patients = patient_df['patient_id'].values
patient_labels = patient_df['label'].values
print(f"Unique patients: {len(unique_patients)}")
print(f"  With murmur: {patient_labels.sum()}")
print(f"  Without murmur: {(patient_labels == 0).sum()}")

train_patients, test_patients = train_test_split(
    unique_patients,
    test_size=0.2,
    stratify=patient_labels,
    random_state=42
)

train_mask = np.isin(patient_ids_filtered, train_patients)
test_mask = np.isin(patient_ids_filtered, test_patients)

# Split data  - not normalized yet - will normalize per fold
X_train_full = X_filtered[train_mask]
y_train_full = y_encoded[train_mask]
patient_ids_train = patient_ids_filtered[train_mask]
locations_train = locations_filtered[train_mask]

X_test = X_filtered[test_mask]
y_test = y_encoded[test_mask]
patient_ids_test = patient_ids_filtered[test_mask]
locations_test = locations_filtered[test_mask]

print(f"\nTraining set: {X_train_full.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")



Filtered dataset (excluding Unknown): 3007 samples

Encoded labels - Present (1): 616, Absent (0): 2391
Unique patients: 874
  With murmur: 179
  Without murmur: 695

Training set: 2396 samples
Test set: 611 samples


In [9]:

def build_mel_cnn(input_shape):
    #matches model 1
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1: Low-level spectral features
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 2: Mid-level frequency patterns
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 3: High-level features
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.GlobalAveragePooling2D(),


        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])

    return model


input_shape = X_train_full.shape[1:]  # (128, 79, 1)
model = build_mel_cnn(input_shape)
model.summary()

print(f"\nInput shape: {input_shape}")
print(f"Output: 1 (binary - sigmoid)")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 79, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 79, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 128, 79, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 39, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 39, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 39, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 39, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 64, 39, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 19, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 19, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 19, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 19, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 32, 19, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,889 (398.00 KB)

 Trainable params: 101,441 (396.25 KB)

 Non-trainable params: 448 (1.75 KB)


Input shape: (128, 79, 1)
Output: 1 (binary - sigmoid)


In [10]:
def normalize_data(X_train, X_val):
    mean = X_train.mean()
    std = X_train.std()
    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std
    return X_train_norm, X_val_norm, mean, std


def aggregate_to_patient_level(preds, labels, patient_ids):
    # Aggregate recording-level predictions to patient level using MAX probability across recordings
    df = pd.DataFrame({
        'patient_id': patient_ids,
        'pred': preds,
        'label': labels
    })

    patient_agg = df.groupby('patient_id').agg({
        'pred': 'max',   # Max probability across recordings
        'label': 'max'   # Should be same for all recordings of a patient
    }).reset_index()

    return patient_agg


def compute_metrics(y_true, y_pred_proba, threshold):

    y_pred = (y_pred_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    try:
        auc = roc_auc_score(y_true, y_pred_proba)
    except:
        auc = 0.5

    try:
        auprc = average_precision_score(y_true, y_pred_proba)
    except:
        auprc = 0.0

    try:
        f1 = f1_score(y_true, y_pred)
    except:
        f1 = 0.0

    return {
        'auc': auc,
        'auprc': auprc,
        'f1': f1,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'threshold': threshold,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn
    }


def get_predictions(model, X, batch_size=32):
    preds = model.predict(X, batch_size=batch_size, verbose=0)
    return preds.squeeze()


In [11]:
# crossvalidation

train_patient_df = pd.DataFrame({
    'patient_id': patient_ids_train,
    'label': y_train_full
}).drop_duplicates(subset='patient_id')

train_unique_patients = train_patient_df['patient_id'].values
train_patient_labels = train_patient_df['label'].values

print(f"Cross-validation on {len(train_unique_patients)} patients")
print(f"  With murmur: {train_patient_labels.sum()}")
print(f"  Without murmur: {(train_patient_labels == 0).sum()}")


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)


cv_val_probs = []
cv_val_labels = []
cv_val_patient_ids = []
fold_metrics = []

input_shape = X_train_full.shape[1:]

for fold, (train_idx, val_idx) in enumerate(skf.split(train_unique_patients, train_patient_labels)):

    print(f'FOLD {fold+1}/{N_FOLDS}')

    # Get patient IDs for this fold
    fold_train_pids = train_unique_patients[train_idx]
    fold_val_pids = train_unique_patients[val_idx]

    fold_train_mask = np.isin(patient_ids_train, fold_train_pids)
    fold_val_mask = np.isin(patient_ids_train, fold_val_pids)

    X_fold_train = X_train_full[fold_train_mask]
    y_fold_train = y_train_full[fold_train_mask]
    X_fold_val = X_train_full[fold_val_mask]
    y_fold_val = y_train_full[fold_val_mask]
    fold_val_patient_ids = patient_ids_train[fold_val_mask]

    print(f'Train: {len(X_fold_train)} recordings from {len(fold_train_pids)} patients')
    print(f'Val:   {len(X_fold_val)} recordings from {len(fold_val_pids)} patients')

    X_fold_train_norm, X_fold_val_norm, fold_mean, fold_std = normalize_data(
        X_fold_train, X_fold_val
    )

    # Compute class weights for this fold
    n_neg = (y_fold_train == 0).sum()
    n_pos = (y_fold_train == 1).sum()
    class_weight = {0: 1.0, 1: n_neg / n_pos}
    print(f'Class weight for Present: {class_weight[1]:.2f}')

    # Build fresh model for each fold
    model = build_mel_cnn(input_shape)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=0
        )
    ]

    # Train
    history = model.fit(
        X_fold_train_norm, y_fold_train,
        validation_data=(X_fold_val_norm, y_fold_val),  # Explicit val set
        epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=0
    )

    print(f'Stopped at epoch {len(history.history["loss"])}')
    print(f'Final train loss: {history.history["loss"][-1]:.4f}')
    print(f'Final val loss: {history.history["val_loss"][-1]:.4f}')

    # Get validation predictions (recording-level)
    val_probs = get_predictions(model, X_fold_val_norm)

    val_patient_df = aggregate_to_patient_level(
        val_probs, y_fold_val, fold_val_patient_ids
    )

    cv_val_probs.extend(val_patient_df['pred'].values)
    cv_val_labels.extend(val_patient_df['label'].values)
    cv_val_patient_ids.extend(val_patient_df['patient_id'].values)

    fold_auroc = roc_auc_score(val_patient_df['label'], val_patient_df['pred'])
    fold_auprc = average_precision_score(val_patient_df['label'], val_patient_df['pred'])

    fold_metrics.append({
        'fold': fold + 1,
        'auroc': fold_auroc,
        'auprc': fold_auprc,
        'n_patients': len(val_patient_df),
        'n_positive': val_patient_df['label'].sum()
    })

    print(f'\nFold {fold+1} Patient-Level Results:')
    print(f'  AUROC: {fold_auroc:.4f}')
    print(f'  AUPRC: {fold_auprc:.4f}')

    del model
    keras.backend.clear_session()

cv_val_probs = np.array(cv_val_probs)
cv_val_labels = np.array(cv_val_labels)

print('CROSS-VALIDATION SUMMARY')


fold_metrics_df = pd.DataFrame(fold_metrics)
print(f"\nPer-fold results:")
print(fold_metrics_df.to_string(index=False))

mean_auroc = fold_metrics_df['auroc'].mean()
std_auroc = fold_metrics_df['auroc'].std()
mean_auprc = fold_metrics_df['auprc'].mean()
std_auprc = fold_metrics_df['auprc'].std()

print(f"\nMean AUROC: {mean_auroc:.4f} ± {std_auroc:.4f}")
print(f"Mean AUPRC: {mean_auprc:.4f} ± {std_auprc:.4f}")
print(f"\nPooled validation predictions: {len(cv_val_probs)} patients")

Cross-validation on 699 patients
  With murmur: 143
  Without murmur: 556
FOLD 1/5
Train: 1932 recordings from 559 patients
Val:   464 recordings from 140 patients
Class weight for Present: 3.82
Epoch 36: early stopping
Restoring model weights from the end of the best epoch: 21.
Stopped at epoch 36
Final train loss: 0.8517
Final val loss: 0.5033

Fold 1 Patient-Level Results:
  AUROC: 0.6747
  AUPRC: 0.5183
FOLD 2/5
Train: 1904 recordings from 559 patients
Val:   492 recordings from 140 patients
Class weight for Present: 3.84
Epoch 32: early stopping
Restoring model weights from the end of the best epoch: 17.
Stopped at epoch 32
Final train loss: 0.9225
Final val loss: 0.4596

Fold 2 Patient-Level Results:
  AUROC: 0.8263
  AUPRC: 0.7421
FOLD 3/5
Train: 1924 recordings from 559 patients
Val:   472 recordings from 140 patients
Class weight for Present: 3.91
Epoch 43: early stopping
Restoring model weights from the end of the best epoch: 28.
Stopped at epoch 43
Final train loss: 0.8517
F

In [12]:

from sklearn.metrics import roc_curve

print("Selecting optimal threshold using pooled CV validation predictions...")

# F1-optimized threshold
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(cv_val_labels, cv_val_probs)
f1_scores = 2 * (precision_curve * recall_curve) / (precision_curve + recall_curve + 1e-8)
best_f1_idx = np.argmax(f1_scores[:-1])  # Exclude last point

optimal_threshold = pr_thresholds[best_f1_idx]
best_precision = precision_curve[best_f1_idx]
best_recall = recall_curve[best_f1_idx]
best_f1 = f1_scores[best_f1_idx]

print(f"\n--- F1-Optimized Threshold ---")
print(f"Selected threshold: {optimal_threshold:.4f}")
print(f"Precision: {best_precision:.4f}")
print(f"Recall (Sensitivity): {best_recall:.4f}")
print(f"F1 Score: {best_f1:.4f}")

print(f"OPTIMAL THRESHOLD: {optimal_threshold:.4f}")



thresholds_dict = {
    'f1_optimized': optimal_threshold,
    'primary': optimal_threshold
}

Selecting optimal threshold using pooled CV validation predictions...

--- F1-Optimized Threshold ---
Selected threshold: 0.5030
Precision: 0.6907
Recall (Sensitivity): 0.4685
F1 Score: 0.5583
OPTIMAL THRESHOLD: 0.5030


In [13]:
# training final model
X_train_mean = X_train_full.mean()
X_train_std = X_train_full.std()

X_train_norm = (X_train_full - X_train_mean) / X_train_std
X_test_norm = (X_test - X_train_mean) / X_train_std

# Class weights
n_neg = (y_train_full == 0).sum()
n_pos = (y_train_full == 1).sum()
class_weight = {0: 1.0, 1: n_neg / n_pos}

# Build final model
final_model = build_mel_cnn(input_shape)

final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# For final training, we need a validation set for early stopping

val_split_patients, _ = train_test_split(
    train_unique_patients,
    test_size=0.9,
    stratify=train_patient_labels,
    random_state=99
)

final_val_mask = np.isin(patient_ids_train, val_split_patients)
final_train_mask = ~final_val_mask

X_final_train = X_train_norm[final_train_mask]
y_final_train = y_train_full[final_train_mask]
X_final_val = X_train_norm[final_val_mask]
y_final_val = y_train_full[final_val_mask]

print(f"\nFinal training split:")
print(f"  Train: {len(X_final_train)} recordings")
print(f"  Val (for early stopping): {len(X_final_val)} recordings")

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'm2/model2_final.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

# Train final model
history = final_model.fit(
    X_final_train, y_final_train,
    validation_data=(X_final_val, y_final_val),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

print(f"\nTraining complete!")
print(f"Stopped at epoch: {len(history.history['loss'])}")
print(f"Best val loss: {min(history.history['val_loss']):.4f}")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Model Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set_title('Model Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('m2/training_history.png', dpi=150)
plt.show()


Final training split:
  Train: 2158 recordings
  Val (for early stopping): 238 recordings
Epoch 1/100
68/68 ━━━━━━━━━━━━━━━━━━━━ 17s 131ms/step - accuracy: 0.4676 - loss: 1.1197 - val_accuracy: 0.7815 - val_loss: 0.6862 - learning_rate: 1.0000e-04
Epoch 2/100
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.5598 - loss: 1.0824 - val_accuracy: 0.7941 - val_loss: 0.6089 - learning_rate: 1.0000e-04
Epoch 3/100
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5825 - loss: 1.0719 - val_accuracy: 0.7941 - val_loss: 0.6075 - learning_rate: 1.0000e-04
Epoch 4/100
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5561 - loss: 1.0608 - val_accuracy: 0.7941 - val_loss: 0.5899 - learning_rate: 1.0000e-04
Epoch 5/100
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6386 - loss: 1.0294 - val_accuracy: 0.7941 - val_loss: 0.5612 - learning_rate: 1.0000e-04
Epoch 6/100
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6381 - loss: 1.0209 - val_accuracy: 0.7983 - val_loss: 0.5597

In [14]:

print("FINAL EVALUATION ON HELD-OUT TEST SET")


# Get predictions on test set
test_probs = get_predictions(final_model, X_test_norm)

print(f"\nTest set: {len(X_test)} recordings from {len(np.unique(patient_ids_test))} patients")


print(f"\n{'='*40}")
print("RECORDING-LEVEL METRICS")
print(f"{'='*40}")

rec_metrics = compute_metrics(y_test, test_probs, optimal_threshold)
print(f"\nAt threshold = {optimal_threshold:.4f}:")
print(f"  Sensitivity: {rec_metrics['sensitivity']:.4f}")
print(f"  Specificity: {rec_metrics['specificity']:.4f}")
print(f"  F1 Score:    {rec_metrics['f1']:.4f}")
print(f"  AUROC:       {rec_metrics['auc']:.4f}")
print(f"  AUPRC:       {rec_metrics['auprc']:.4f}")

# --- Patient-Level Metrics (PRIMARY) ---
print(f"\n{'='*40}")
print("PATIENT-LEVEL METRICS (PRIMARY)")
print(f"{'='*40}")

# Aggregate to patient level
test_patient_df = aggregate_to_patient_level(test_probs, y_test, patient_ids_test)

patient_probs = test_patient_df['pred'].values
patient_labels = test_patient_df['label'].values

print(f"\nTest patients: {len(test_patient_df)}")
print(f"  With murmur: {patient_labels.sum()}")
print(f"  Without murmur: {(patient_labels == 0).sum()}")

# Compute patient-level metrics
patient_metrics = compute_metrics(patient_labels, patient_probs, optimal_threshold)

print(f"\nAt threshold = {optimal_threshold:.4f}:")
print(f"  Sensitivity: {patient_metrics['sensitivity']:.4f}")
print(f"  Specificity: {patient_metrics['specificity']:.4f}")
print(f"  F1 Score:    {patient_metrics['f1']:.4f}")
print(f"  AUROC:       {patient_metrics['auc']:.4f}")
print(f"  AUPRC:       {patient_metrics['auprc']:.4f}")

print(f"\nConfusion Matrix (Patient-Level):")
print(f"  TP={patient_metrics['tp']}, FP={patient_metrics['fp']}")
print(f"  FN={patient_metrics['fn']}, TN={patient_metrics['tn']}")

# --- Compare Different Thresholds ---
print(f"\n{'='*40}")
print("COMPARISON OF THRESHOLDS")
print(f"{'='*40}")

threshold_comparison = []
for name, thresh in thresholds_dict.items():
    metrics = compute_metrics(patient_labels, patient_probs, thresh)
    threshold_comparison.append({
        'Threshold': name,
        'Value': thresh,
        'Sensitivity': metrics['sensitivity'],
        'Specificity': metrics['specificity'],
        'F1': metrics['f1']
    })

threshold_df = pd.DataFrame(threshold_comparison)
print(threshold_df.to_string(index=False))

# --- Visualization ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. ROC Curve (Patient-Level)
fpr_test, tpr_test, _ = roc_curve(patient_labels, patient_probs)
axes[0, 0].plot(fpr_test, tpr_test, 'b-', linewidth=2,
                label=f'ROC (AUC={patient_metrics["auc"]:.3f})')
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0, 0].scatter(1-patient_metrics['specificity'], patient_metrics['sensitivity'],
                   color='red', s=100, zorder=5,
                   label=f'Operating point (t={optimal_threshold:.3f})')
axes[0, 0].axhline(y=MIN_SENSITIVITY, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0, 0].set_ylabel('True Positive Rate (Sensitivity)')
axes[0, 0].set_title('ROC Curve (Patient-Level)')
axes[0, 0].legend(loc='lower right')
axes[0, 0].grid(True, alpha=0.3)

# 2. Precision-Recall Curve (Patient-Level)
prec_test, rec_test, _ = precision_recall_curve(patient_labels, patient_probs)
axes[0, 1].plot(rec_test, prec_test, 'b-', linewidth=2,
                label=f'PR (AUC={patient_metrics["auprc"]:.3f})')
# Find precision at operating point
test_preds = (patient_probs >= optimal_threshold).astype(int)
op_precision = patient_metrics['tp'] / (patient_metrics['tp'] + patient_metrics['fp']) if (patient_metrics['tp'] + patient_metrics['fp']) > 0 else 0
axes[0, 1].scatter(patient_metrics['sensitivity'], op_precision,
                   color='red', s=100, zorder=5,
                   label=f'Operating point')
axes[0, 1].set_xlabel('Recall (Sensitivity)')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].set_title('Precision-Recall Curve (Patient-Level)')
axes[0, 1].legend(loc='lower left')
axes[0, 1].grid(True, alpha=0.3)

# 3. Confusion Matrix
cm = confusion_matrix(patient_labels, test_preds)
im = axes[1, 0].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
axes[1, 0].set_title(f'Confusion Matrix (t={optimal_threshold:.3f})')
plt.colorbar(im, ax=axes[1, 0])
axes[1, 0].set_xticks([0, 1])
axes[1, 0].set_yticks([0, 1])
axes[1, 0].set_xticklabels(['Absent', 'Present'])
axes[1, 0].set_yticklabels(['Absent', 'Present'])
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('True')

for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max()/2 else 'black'
        axes[1, 0].text(j, i, format(cm[i, j], 'd'),
                        ha='center', va='center', color=color, fontsize=14)

# 4. Probability Distribution
axes[1, 1].hist(patient_probs[patient_labels == 0], bins=20, alpha=0.6,
                label='Absent', color='blue', density=True)
axes[1, 1].hist(patient_probs[patient_labels == 1], bins=20, alpha=0.6,
                label='Present', color='red', density=True)
axes[1, 1].axvline(x=optimal_threshold, color='green', linestyle='-',
                   linewidth=2, label=f'Threshold={optimal_threshold:.3f}')
axes[1, 1].set_xlabel('Predicted Probability')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Probability Distribution (Patient-Level)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('m2/test_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Final Summary ---
print(f"\n{'='*60}")
print("MODEL 2 FINAL RESULTS SUMMARY")
print(f"{'='*60}")
print(f"Model: 2D CNN on Mel Spectrograms")
print(f"Input shape: {input_shape}")
print(f"Threshold: {optimal_threshold:.4f} (sensitivity-optimized)")
print(f"\nPatient-Level Test Performance:")
print(f"  AUROC:       {patient_metrics['auc']:.4f}")
print(f"  AUPRC:       {patient_metrics['auprc']:.4f}")
print(f"  Sensitivity: {patient_metrics['sensitivity']:.4f}")
print(f"  Specificity: {patient_metrics['specificity']:.4f}")
print(f"  F1 Score:    {patient_metrics['f1']:.4f}")

FINAL EVALUATION ON HELD-OUT TEST SET

Test set: 611 recordings from 175 patients

RECORDING-LEVEL METRICS

At threshold = 0.5030:
  Sensitivity: 0.3171
  Specificity: 0.9959
  F1 Score:    0.4756
  AUROC:       0.7319
  AUPRC:       0.6054

PATIENT-LEVEL METRICS (PRIMARY)

Test patients: 175
  With murmur: 36
  Without murmur: 139

At threshold = 0.5030:
  Sensitivity: 0.3889
  Specificity: 0.9856
  F1 Score:    0.5385
  AUROC:       0.7398
  AUPRC:       0.6262

Confusion Matrix (Patient-Level):
  TP=14, FP=2
  FN=22, TN=137

COMPARISON OF THRESHOLDS
   Threshold    Value  Sensitivity  Specificity       F1
f1_optimized 0.502992     0.388889     0.985612 0.538462
     primary 0.502992     0.388889     0.985612 0.538462

MODEL 2 FINAL RESULTS SUMMARY
Model: 2D CNN on Mel Spectrograms
Input shape: (128, 79, 1)
Threshold: 0.5030 (sensitivity-optimized)

Patient-Level Test Performance:
  AUROC:       0.7398
  AUPRC:       0.6262
  Sensitivity: 0.3889
  Specificity: 0.9856
  F1 Score:    0

In [15]:
print("Saving model and results...")

# 1. Save the final model
final_model.save('m2/model2_final.keras')
print("✓ Saved: m2/model2_final.keras")

# 2. Save preprocessing parameters (simplified - only F1 threshold)
preprocessing_params = {
    'X_mean': float(X_train_mean),
    'X_std': float(X_train_std),
    'sample_rate': SAMPLE_RATE,
    'n_mels': N_MELS,
    'hop_length': HOP_LENGTH,
    'n_fft': N_FFT,
    'duration': DURATION,
    'optimal_threshold': float(optimal_threshold)
}

np.save('m2/model2_preprocessing_params.npy', preprocessing_params)
print("✓ Saved: m2/model2_preprocessing_params.npy")

# 3. Save comprehensive results
results = {
    'model_name': 'Model 2: 2D CNN on Mel Spectrograms',
    'architecture': {
        'input_shape': list(input_shape),
        'conv_blocks': 3,
        'filters': [32, 64, 128],
        'output': 'sigmoid (binary)'
    },
    'training': {
        'n_folds': N_FOLDS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'epochs_max': NUM_EPOCHS,
        'train_recordings': int(len(X_train_full)),
        'train_patients': int(len(train_unique_patients))
    },
    'cv_results': {
        'mean_auroc': float(mean_auroc),
        'std_auroc': float(std_auroc),
        'mean_auprc': float(mean_auprc),
        'std_auprc': float(std_auprc)
    },
    'threshold': {
        'method': 'f1_optimized',
        'value': float(optimal_threshold)
    },
    'test_results': {
        'recording_level': {
            'n_recordings': int(len(X_test)),
            'auroc': float(rec_metrics['auc']),
            'auprc': float(rec_metrics['auprc']),
            'sensitivity': float(rec_metrics['sensitivity']),
            'specificity': float(rec_metrics['specificity']),
            'f1': float(rec_metrics['f1'])
        },
        'patient_level': {
            'n_patients': int(len(test_patient_df)),
            'n_positive': int(patient_labels.sum()),
            'n_negative': int((patient_labels == 0).sum()),
            'auroc': float(patient_metrics['auc']),
            'auprc': float(patient_metrics['auprc']),
            'sensitivity': float(patient_metrics['sensitivity']),
            'specificity': float(patient_metrics['specificity']),
            'f1': float(patient_metrics['f1']),
            'tp': int(patient_metrics['tp']),
            'fp': int(patient_metrics['fp']),
            'tn': int(patient_metrics['tn']),
            'fn': int(patient_metrics['fn'])
        }
    }
}

import json
with open('m2/model2_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(" Saved: m2/model2_results.json")

# 4. Save fold-level metrics
fold_metrics_df.to_csv('m2/model2_cv_fold_metrics.csv', index=False)
print(" Saved: m2/model2_cv_fold_metrics.csv")

# 5. Save test predictions for later analysis
test_predictions_df = pd.DataFrame({
    'patient_id': test_patient_df['patient_id'],
    'true_label': test_patient_df['label'],
    'pred_probability': test_patient_df['pred'],
    'pred_label': (test_patient_df['pred'] >= optimal_threshold).astype(int)
})
test_predictions_df.to_csv('m2/model2_test_predictions.csv', index=False)
print(" Saved: m2/model2_test_predictions.csv")

# --- Print Final Summary ---
print(f"\n{'='*60}")
print("MODEL 2 COMPLETE - FINAL SUMMARY")


print(f"\nCross-Validation Results:")
print(f"   AUROC: {mean_auroc:.4f} ± {std_auroc:.4f}")
print(f"   AUPRC: {mean_auprc:.4f} ± {std_auprc:.4f}")

print(f"\nSelected Threshold: {optimal_threshold:.4f}")
print(f"   (F1-optimized)")

print(f"\n Test Set Performance (Patient-Level):")
print(f"   AUROC:       {patient_metrics['auc']:.4f}")
print(f"   AUPRC:       {patient_metrics['auprc']:.4f}")
print(f"   Sensitivity: {patient_metrics['sensitivity']:.4f} ({patient_metrics['tp']}/{patient_metrics['tp']+patient_metrics['fn']} murmurs detected)")
print(f"   Specificity: {patient_metrics['specificity']:.4f}")
print(f"   F1 Score:    {patient_metrics['f1']:.4f}")


Saving model and results...
✓ Saved: m2/model2_final.keras
✓ Saved: m2/model2_preprocessing_params.npy
 Saved: m2/model2_results.json
 Saved: m2/model2_cv_fold_metrics.csv
 Saved: m2/model2_test_predictions.csv

MODEL 2 COMPLETE - FINAL SUMMARY

Cross-Validation Results:
   AUROC: 0.7776 ± 0.0804
   AUPRC: 0.6249 ± 0.1203

Selected Threshold: 0.5030
   (F1-optimized)

 Test Set Performance (Patient-Level):
   AUROC:       0.7398
   AUPRC:       0.6262
   Sensitivity: 0.3889 (14/36 murmurs detected)
   Specificity: 0.9856
   F1 Score:    0.5385


In [16]:
# class_weights = dict(zip(
#     range(num_classes),
#     len(y_train) / (num_classes * np.bincount(y_train))
# ))
# print(f"\nClass weights: {class_weights}")
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# callbacks = [
#     tf.keras.callbacks.EarlyStopping(
#         monitor='val_loss',
#         patience=15,
#         restore_best_weights=True,
#         verbose=1
#     ),
#     tf.keras.callbacks.ReduceLROnPlateau(
#         monitor='val_loss',
#         factor=0.5,
#         patience=5,
#         min_lr=1e-6,
#         verbose=1
#     ),
#     tf.keras.callbacks.ModelCheckpoint(
#         'm2/model2_mel_cnn_best.keras',
#         monitor='val_loss',
#         save_best_only=True,
#         verbose=1
#     )
# ]

# # Train
# print("\nTraining Model 2: Mel Spectrogram CNN...")
# history = model.fit(
#     X_train, y_train,
#     validation_split=0.2,
#     epochs=100,
#     batch_size=32,
#     class_weight=class_weights,
#     callbacks=callbacks,
#     verbose=1
# )

In [17]:
# def plot_training_history(history):
#     """Plot training curves"""
#     fig, axes = plt.subplots(1, 2, figsize=(12, 4))

#     # Loss
#     axes[0].plot(history.history['loss'], label='Train')
#     axes[0].plot(history.history['val_loss'], label='Validation')
#     axes[0].set_title('Model Loss')
#     axes[0].set_xlabel('Epoch')
#     axes[0].set_ylabel('Loss')
#     axes[0].legend()

#     # Accuracy
#     axes[1].plot(history.history['accuracy'], label='Train')
#     axes[1].plot(history.history['val_accuracy'], label='Validation')
#     axes[1].set_title('Model Accuracy')
#     axes[1].set_xlabel('Epoch')
#     axes[1].set_ylabel('Accuracy')
#     axes[1].legend()

#     plt.tight_layout()
#     plt.savefig('m2/model2_training_history.png', dpi=150)
#     plt.show()

# plot_training_history(history)

# # Test set evaluation
# print("\n" + "="*50)
# print("MODEL 2 EVALUATION - Test Set")
# print("="*50)

# y_pred_proba = model.predict(X_test)
# y_pred = np.argmax(y_pred_proba, axis=1)

# print("\nClassification Report:")
# print(classification_report(y_test, y_pred, target_names=le.classes_))

# # Individual metrics
# precision = precision_score(y_test, y_pred, average='weighted')
# recall = recall_score(y_test, y_pred, average='weighted')
# f1 = f1_score(y_test, y_pred, average='weighted')

# print(f"\nWeighted Precision: {precision:.4f}")
# print(f"Weighted Recall: {recall:.4f}")
# print(f"Weighted F1-Score: {f1:.4f}")

In [18]:
# cm = confusion_matrix(y_test, y_pred)
# plt.figure(figsize=(8, 6))
# plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
# plt.title('Model 2: Confusion Matrix')
# plt.colorbar()
# tick_marks = np.arange(len(le.classes_))
# plt.xticks(tick_marks, le.classes_, rotation=45)
# plt.yticks(tick_marks, le.classes_)
# plt.xlabel('Predicted')
# plt.ylabel('True')

# for i in range(cm.shape[0]):
#     for j in range(cm.shape[1]):
#         plt.text(j, i, format(cm[i, j], 'd'),
#                 ha="center", va="center",
#                 color="white" if cm[i, j] > cm.max()/2 else "black")

# plt.tight_layout()
# plt.savefig('m2/model2_confusion_matrix.png', dpi=150)
# plt.show()

In [19]:
# model.save('m2/model2_mel_spectrogram_cnn.keras')

# preprocessing_params = {
#     'X_mean': X_mean,
#     'X_std': X_std,
#     'label_encoder_classes': le.classes_.tolist(),
#     'sample_rate': SAMPLE_RATE,
#     'n_mels': N_MELS,
#     'hop_length': HOP_LENGTH,
#     'n_fft': N_FFT,
#     'duration': DURATION
# }
# np.save('m2/model2_preprocessing_params.npy', preprocessing_params)

# print("\nModel 2 training complete!")
# print("Saved: model2_mel_spectrogram_cnn.keras")
# print("Saved: model2_preprocessing_params.npy")

In [20]:
# from tensorflow.keras import regularizers
# def build_mel_spectrogram_cnn_regularized(input_shape, num_classes):
#     """
#     2D CNN with stronger regularization to prevent overfitting.
#     """
#     l2_reg = 0.001  # L2 regularization strength

#     model = models.Sequential([
#         layers.Input(shape=input_shape),

#         # Data Augmentation (only applied during training)
#         layers.GaussianNoise(0.1),  # Add noise for robustness

#         # Block 1: Reduced filters + stronger regularization
#         layers.Conv2D(16, (3, 3), padding='same', kernel_regularizer=regularizers.l2(l2_reg)),
#         layers.BatchNormalization(),
#         layers.ReLU(),
#         layers.MaxPooling2D((2, 2)),
#         layers.Dropout(0.3),

#         # Block 2
#         layers.Conv2D(32, (3, 3), padding='same', kernel_regularizer=regularizers.l2(l2_reg)),
#         layers.BatchNormalization(),
#         layers.ReLU(),
#         layers.MaxPooling2D((2, 2)),
#         layers.Dropout(0.3),

#         # Block 3
#         layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=regularizers.l2(l2_reg)),
#         layers.BatchNormalization(),
#         layers.ReLU(),
#         layers.MaxPooling2D((2, 2)),
#         layers.Dropout(0.4),

#         # Block 4
#         layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=regularizers.l2(l2_reg)),
#         layers.BatchNormalization(),
#         layers.ReLU(),
#         layers.GlobalAveragePooling2D(),

#         # Simplified classification head
#         layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
#         layers.Dropout(0.5),
#         layers.Dense(num_classes, activation='softmax')
#     ])

#     return model

# model = build_mel_spectrogram_cnn_regularized(input_shape, num_classes)
# model.summary()

In [21]:
# def apply_spec_augment(spectrogram, freq_mask_param=15, time_mask_param=20, num_masks=2):
#     """
#     Apply SpecAugment: random frequency and time masking.

#     Args:
#         spectrogram: input spectrogram (n_mels, time_steps, 1)
#         freq_mask_param: max width of frequency mask
#         time_mask_param: max width of time mask
#         num_masks: number of masks to apply
#     Returns:
#         augmented spectrogram
#     """
#     spec = spectrogram.copy()
#     n_mels, time_steps, _ = spec.shape

#     for _ in range(num_masks):
#         # Frequency masking
#         f = np.random.randint(0, freq_mask_param)
#         f0 = np.random.randint(0, n_mels - f)
#         spec[f0:f0+f, :, :] = 0

#         # Time masking
#         t = np.random.randint(0, time_mask_param)
#         t0 = np.random.randint(0, time_steps - t)
#         spec[:, t0:t0+t, :] = 0

#     return spec

# # Create augmented training set
# print("Applying SpecAugment to training data...")
# X_train_augmented = np.array([apply_spec_augment(x) for x in X_train])

# # Combine original and augmented data
# X_train_combined = np.concatenate([X_train, X_train_augmented], axis=0)
# y_train_combined = np.concatenate([y_train, y_train], axis=0)

# print(f"Original training size: {len(X_train)}")
# print(f"Augmented training size: {len(X_train_combined)}")

In [22]:
# class_weights = dict(zip(
#     range(num_classes),
#     len(y_train_combined) / (num_classes * np.bincount(y_train_combined))
# ))
# print(f"\nClass weights: {class_weights}")

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.000005),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# callbacks = [
#     tf.keras.callbacks.EarlyStopping(
#         monitor='val_loss',
#         patience=20,
#         restore_best_weights=True,
#         verbose=1
#     ),
#     tf.keras.callbacks.ReduceLROnPlateau(
#         monitor='val_loss',
#         factor=0.5,
#         patience=7,
#         min_lr=1e-7,
#         verbose=1
#     ),
#     tf.keras.callbacks.ModelCheckpoint(
#         'm2/model2_mel_cnn_best.keras',
#         monitor='val_loss',
#         save_best_only=True,
#         verbose=1
#     )
# ]

# print("\nTraining Model 2 (Regularized): Mel Spectrogram CNN...")
# history = model.fit(
#     X_train_combined, y_train_combined,
#     validation_split=0.2,
#     epochs=100,
#     batch_size=64,
#     class_weight=class_weights,
#     callbacks=callbacks,
#     verbose=1
# )

In [23]:
# print("\n" + "="*50)
# print("MODEL 2 EVALUATION (Regularized) - Test Set")
# print("="*50)

# y_pred_proba = model.predict(X_test)
# y_pred = np.argmax(y_pred_proba, axis=1)

# print("\nClassification Report:")
# print(classification_report(y_test, y_pred, target_names=le.classes_))

# # Individual metrics
# precision = precision_score(y_test, y_pred, average='weighted')
# recall = recall_score(y_test, y_pred, average='weighted')
# f1 = f1_score(y_test, y_pred, average='weighted')

# print(f"\nWeighted Precision: {precision:.4f}")
# print(f"Weighted Recall: {recall:.4f}")
# print(f"Weighted F1-Score: {f1:.4f}")

In [24]:
# cm = confusion_matrix(y_test, y_pred)
# plt.figure(figsize=(8, 6))
# plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
# plt.title('Model 2: Confusion Matrix')
# plt.colorbar()
# tick_marks = np.arange(len(le.classes_))
# plt.xticks(tick_marks, le.classes_, rotation=45)
# plt.yticks(tick_marks, le.classes_)
# plt.xlabel('Predicted')
# plt.ylabel('True')

# for i in range(cm.shape[0]):
#     for j in range(cm.shape[1]):
#         plt.text(j, i, format(cm[i, j], 'd'),
#                 ha="center", va="center",
#                 color="white" if cm[i, j] > cm.max()/2 else "black")

# plt.tight_layout()
# plt.savefig('m2/model2_confusion_matrix.png', dpi=150)
# plt.show()

In [25]:
# BASE_PATH = '1.0.3/training_data'
# SAMPLE_RATE = 4000
# N_MELS = 128
# HOP_LENGTH = 256
# N_FFT = 1024
# DURATION = 5

# cohort = pd.read_csv('cohort.csv')
# print(f"Total patients: {len(cohort)}")
# print(f"Murmur distribution:\n{cohort['Murmur'].value_counts()}")
# print(f"Outcome distribution:\n{cohort['Outcome'].value_counts()}")


# def extract_mel_spectrogram(wav_path, sr=SAMPLE_RATE, n_mels=N_MELS, duration=DURATION):
#     try:
#         signal, sr = librosa.load(wav_path, sr=sr, duration=duration)
#         target_length = sr * duration
#         if len(signal) < target_length:
#             signal = np.pad(signal, (0, target_length - len(signal)), mode='constant')
#         else:
#             signal = signal[:target_length]

#         mel_spec = librosa.feature.melspectrogram(
#             y=signal,
#             sr=sr,
#             n_mels=n_mels,
#             n_fft=N_FFT,
#             hop_length=HOP_LENGTH,
#             fmin=20,
#             fmax=1000
#         )
#         mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
#         return mel_spec_db
#     except Exception as e:
#         print(f"Error processing {wav_path}: {e}")
#         return None


# def process_all_recordings(cohort_df, base_path):
#     spectrograms = []
#     labels_murmur = []
#     labels_outcome = []
#     patient_ids = []
#     locations = []

#     for idx, row in tqdm(cohort_df.iterrows(), total=len(cohort_df), desc="Processing audio"):
#         patient_id = row['patient_id']
#         try:
#             recordings = eval(row['recordings'])
#         except:
#             continue

#         for rec in recordings:
#             wav_file = rec['wav_file']
#             location = rec['location']
#             wav_path = os.path.join(base_path, wav_file)

#             if os.path.exists(wav_path):
#                 mel_spec = extract_mel_spectrogram(wav_path)
#                 if mel_spec is not None:
#                     spectrograms.append(mel_spec)
#                     labels_murmur.append(row['Murmur'])
#                     labels_outcome.append(row['Outcome'])
#                     patient_ids.append(patient_id)
#                     locations.append(location)
#             else:
#                 print(f"File not found: {wav_path}")

#     return (np.array(spectrograms),
#             np.array(labels_murmur),
#             np.array(labels_outcome),
#             np.array(patient_ids),
#             np.array(locations))


# print("Extracting Mel spectrograms...")
# X, y_murmur, y_outcome, patient_ids, locations = process_all_recordings(cohort, BASE_PATH)

# print(f"\nDataset shape: {X.shape}")
# print(f"Unique patients: {len(np.unique(patient_ids))}")
# print(f"Recording locations: {np.unique(locations)}")

# mask = y_murmur != 'Unknown'
# X_filtered = X[mask]
# y_filtered = y_murmur[mask]
# patient_ids_filtered = patient_ids[mask]

# print(f"\nFiltered dataset (excluding Unknown): {X_filtered.shape[0]} samples")

# le = LabelEncoder()
# y_encoded = le.fit_transform(y_filtered)
# print(f"Classes: {le.classes_}")

# X_filtered = X_filtered[..., np.newaxis]

# X_mean = X_filtered.mean()
# X_std = X_filtered.std()
# X_normalized = (X_filtered - X_mean) / X_std

# unique_patients = np.unique(patient_ids_filtered)
# patient_labels = [y_encoded[patient_ids_filtered == p][0] for p in unique_patients]

# train_patients, test_patients = train_test_split(
#     unique_patients,
#     test_size=0.2,
#     stratify=patient_labels,
#     random_state=42
# )

# train_mask = np.isin(patient_ids_filtered, train_patients)
# test_mask = np.isin(patient_ids_filtered, test_patients)

# X_train, X_test = X_normalized[train_mask], X_normalized[test_mask]
# y_train, y_test = y_encoded[train_mask], y_encoded[test_mask]

# print(f"\nTraining set: {X_train.shape[0]} samples")
# print(f"Test set: {X_test.shape[0]} samples")

# print("Training set class distribution:")
# unique, counts = np.unique(y_train, return_counts=True)
# for cls, count in zip(unique, counts):
#     print(f"  Class {cls} ({le.classes_[cls]}): {count} samples ({100 * count / len(y_train):.1f}%)")
# print(f"\nImbalance ratio: {counts.max() / counts.min():.1f}:1")


# def oversample_minority_class(X, y, target_ratio=1.0):
#     unique, counts = np.unique(y, return_counts=True)
#     majority_class = unique[np.argmax(counts)]
#     minority_class = unique[np.argmin(counts)]

#     X_majority = X[y == majority_class]
#     y_majority = y[y == majority_class]
#     X_minority = X[y == minority_class]
#     y_minority = y[y == minority_class]

#     target_minority_size = int(len(X_majority) * target_ratio)

#     X_minority_upsampled, y_minority_upsampled = resample(
#         X_minority, y_minority,
#         replace=True,
#         n_samples=target_minority_size,
#         random_state=42
#     )

#     X_balanced = np.concatenate([X_majority, X_minority_upsampled])
#     y_balanced = np.concatenate([y_majority, y_minority_upsampled])

#     shuffle_idx = np.random.permutation(len(y_balanced))
#     return X_balanced[shuffle_idx], y_balanced[shuffle_idx]


# X_train_balanced, y_train_balanced = oversample_minority_class(X_train, y_train, target_ratio=1.0)

# print(f"\nBalanced training set:")
# unique, counts = np.unique(y_train_balanced, return_counts=True)
# for cls, count in zip(unique, counts):
#     print(f"  Class {cls} ({le.classes_[cls]}): {count} samples ({100 * count / len(y_train_balanced):.1f}%)")

# input_shape = X_train.shape[1:]
# num_classes = len(le.classes_)


# def build_simple_cnn(input_shape, num_classes):
#     model = models.Sequential([
#         layers.Input(shape=input_shape),

#         layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
#         layers.BatchNormalization(),
#         layers.MaxPooling2D((2, 2)),

#         layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
#         layers.BatchNormalization(),
#         layers.MaxPooling2D((2, 2)),

#         layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
#         layers.BatchNormalization(),
#         layers.GlobalAveragePooling2D(),

#         layers.Dense(32, activation='relu'),
#         layers.Dropout(0.3),
#         layers.Dense(num_classes, activation='softmax')
#     ])
#     return model


# model = build_simple_cnn(input_shape, num_classes)
# model.summary()

# class_weights = {
#     0: 0.5,
#     1: 3.0
# }
# print(f"Manual class weights: {class_weights}")

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# callbacks = [
#     tf.keras.callbacks.EarlyStopping(
#         monitor='val_loss',
#         patience=15,
#         restore_best_weights=True,
#         verbose=1
#     ),
#     tf.keras.callbacks.ReduceLROnPlateau(
#         monitor='val_loss',
#         factor=0.5,
#         patience=5,
#         min_lr=1e-6,
#         verbose=1
#     ),
#     tf.keras.callbacks.ModelCheckpoint(
#         'm2/model2_mel_cnn_best.keras',
#         monitor='val_loss',
#         save_best_only=True,
#         verbose=1
#     )
# ]

# print("\nTraining Model 2 on balanced data...")
# history = model.fit(
#     X_train_balanced, y_train_balanced,
#     validation_split=0.2,
#     epochs=100,
#     batch_size=32,
#     class_weight=class_weights,
#     callbacks=callbacks,
#     verbose=1
# )


# def plot_training_history(history):
#     fig, axes = plt.subplots(1, 2, figsize=(12, 4))

#     axes[0].plot(history.history['loss'], label='Train')
#     axes[0].plot(history.history['val_loss'], label='Validation')
#     axes[0].set_title('Model Loss')
#     axes[0].set_xlabel('Epoch')
#     axes[0].set_ylabel('Loss')
#     axes[0].legend()

#     axes[1].plot(history.history['accuracy'], label='Train')
#     axes[1].plot(history.history['val_accuracy'], label='Validation')
#     axes[1].set_title('Model Accuracy')
#     axes[1].set_xlabel('Epoch')
#     axes[1].set_ylabel('Accuracy')
#     axes[1].legend()

#     plt.tight_layout()
#     plt.savefig('m2/model2_training_history.png', dpi=150)
#     plt.show()


# plot_training_history(history)

# y_pred_proba = model.predict(X_test)
# print(f"Prediction probability distribution:")
# print(f"  Min prob class 1: {y_pred_proba[:, 1].min():.4f}")
# print(f"  Max prob class 1: {y_pred_proba[:, 1].max():.4f}")
# print(f"  Mean prob class 1: {y_pred_proba[:, 1].mean():.4f}")

# y_pred_default = np.argmax(y_pred_proba, axis=1)
# print(f"\nDefault threshold (0.5):")
# print(f"  Predicted class 0: {(y_pred_default == 0).sum()}")
# print(f"  Predicted class 1: {(y_pred_default == 1).sum()}")

# y_proba_positive = y_pred_proba[:, 1]
# precision_curve, recall_curve, thresholds = precision_recall_curve(y_test, y_proba_positive)

# f1_scores = 2 * (precision_curve * recall_curve) / (precision_curve + recall_curve + 1e-8)
# optimal_idx = np.argmax(f1_scores)
# optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5

# print(f"Optimal threshold: {optimal_threshold:.4f}")
# print(f"At this threshold - Precision: {precision_curve[optimal_idx]:.4f}, Recall: {recall_curve[optimal_idx]:.4f}")

# plt.figure(figsize=(10, 4))

# plt.subplot(1, 2, 1)
# plt.plot(recall_curve, precision_curve, 'b-', linewidth=2)
# plt.scatter(recall_curve[optimal_idx], precision_curve[optimal_idx],
#             color='red', s=100, zorder=5, label=f'Optimal (thresh={optimal_threshold:.3f})')
# plt.xlabel('Recall')
# plt.ylabel('Precision')
# plt.title('Precision-Recall Curve')
# plt.legend()
# plt.grid(True)

# plt.subplot(1, 2, 2)
# plt.plot(thresholds, f1_scores[:-1], 'g-', linewidth=2)
# plt.axvline(x=optimal_threshold, color='red', linestyle='--', label=f'Optimal={optimal_threshold:.3f}')
# plt.xlabel('Threshold')
# plt.ylabel('F1 Score')
# plt.title('F1 Score vs Threshold')
# plt.legend()
# plt.grid(True)

# plt.tight_layout()
# plt.savefig('m2/model2_threshold_optimization.png', dpi=150)
# plt.show()

# print("\n" + "=" * 50)
# print("DEFAULT THRESHOLD (0.5)")
# print("=" * 50)
# print(classification_report(y_test, y_pred_default, target_names=le.classes_))

# y_pred_optimal = (y_proba_positive >= optimal_threshold).astype(int)
# print("\n" + "=" * 50)
# print(f"OPTIMAL THRESHOLD ({optimal_threshold:.4f})")
# print("=" * 50)
# print(classification_report(y_test, y_pred_optimal, target_names=le.classes_))

# cm = confusion_matrix(y_test, y_pred_optimal)
# plt.figure(figsize=(8, 6))
# plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
# plt.title(f'Model 2: Confusion Matrix (threshold={optimal_threshold:.3f})')
# plt.colorbar()
# tick_marks = np.arange(len(le.classes_))
# plt.xticks(tick_marks, le.classes_, rotation=45)
# plt.yticks(tick_marks, le.classes_)
# plt.xlabel('Predicted')
# plt.ylabel('True')

# for i in range(cm.shape[0]):
#     for j in range(cm.shape[1]):
#         plt.text(j, i, format(cm[i, j], 'd'),
#                  ha="center", va="center",
#                  color="white" if cm[i, j] > cm.max() / 2 else "black")

# plt.tight_layout()
# plt.savefig('m2/model2_confusion_matrix.png', dpi=150)
# plt.show()

# model.save('m2/model2_mel_spectrogram_cnn.keras')

# preprocessing_params = {
#     'X_mean': float(X_mean),
#     'X_std': float(X_std),
#     'label_encoder_classes': le.classes_.tolist(),
#     'sample_rate': SAMPLE_RATE,
#     'n_mels': N_MELS,
#     'hop_length': HOP_LENGTH,
#     'n_fft': N_FFT,
#     'duration': DURATION,
#     'optimal_threshold': float(optimal_threshold)
# }
# np.save('m2/model2_preprocessing_params.npy', preprocessing_params)

# print("\nModel 2 training complete!")
# print("Saved: model2_mel_spectrogram_cnn.keras")
# print("Saved: model2_preprocessing_params.npy")
# print(f"Optimal threshold: {optimal_threshold:.4f}")


In [26]:
# target_recall = 0.70

# recall_diffs = np.abs(recall_curve - target_recall)
# target_idx = np.argmin(recall_diffs)
# recall_threshold = thresholds[target_idx] if target_idx < len(thresholds) else 0.5

# print(f"Threshold for ~{target_recall * 100:.0f}% recall: {recall_threshold:.4f}")
# print(f"At this threshold - Precision: {precision_curve[target_idx]:.4f}, Recall: {recall_curve[target_idx]:.4f}")

# y_pred_recall = (y_proba_positive >= recall_threshold).astype(int)

# print("\n" + "=" * 50)
# print(f"RECALL-OPTIMIZED THRESHOLD ({recall_threshold:.4f})")
# print("=" * 50)
# print(classification_report(y_test, y_pred_recall, target_names=le.classes_))

# cm_recall = confusion_matrix(y_test, y_pred_recall)
# plt.figure(figsize=(8, 6))
# plt.imshow(cm_recall, interpolation='nearest', cmap=plt.cm.Blues)
# plt.title(f'Model 2: Confusion Matrix (recall-optimized, threshold={recall_threshold:.3f})')
# plt.colorbar()
# tick_marks = np.arange(len(le.classes_))
# plt.xticks(tick_marks, le.classes_, rotation=45)
# plt.yticks(tick_marks, le.classes_)
# plt.xlabel('Predicted')
# plt.ylabel('True')

# for i in range(cm_recall.shape[0]):
#     for j in range(cm_recall.shape[1]):
#         plt.text(j, i, format(cm_recall[i, j], 'd'),
#                  ha="center", va="center",
#                  color="white" if cm_recall[i, j] > cm_recall.max() / 2 else "black")

# plt.tight_layout()
# plt.savefig('model2_confusion_matrix_recall_optimized.png', dpi=150)
# plt.show()

# print("\n" + "=" * 50)
# print("MODEL 2 SUMMARY")
# print("=" * 50)
# print(f"Architecture: 2D CNN on Mel Spectrograms")
# print(f"Training samples: {len(X_train_balanced)} (balanced)")
# print(f"Test samples: {len(X_test)}")
# print(f"\nThreshold options:")
# print(f"  F1-optimized:     {optimal_threshold:.4f} (Recall={recall_curve[optimal_idx]:.2f})")
# print(f"  Recall-optimized: {recall_threshold:.4f} (Recall={recall_curve[target_idx]:.2f})")

In [27]:
# print("\n" + "=" * 60)
# print("MODEL 2 FINAL RESULTS - 2D CNN MEL SPECTROGRAM")
# print("=" * 60)

# results = {
#     'model': 'Model 2: 2D CNN Mel Spectrogram',
#     'train_samples': len(X_train_balanced),
#     'test_samples': len(X_test),
#     'input_shape': X_train.shape[1:],
#     'thresholds': {
#         'f1_optimized': {
#             'value': float(optimal_threshold),
#             'precision_present': 0.74,
#             'recall_present': 0.49,
#             'f1_present': 0.59,
#             'accuracy': 0.86
#         },
#         'recall_optimized': {
#             'value': float(recall_threshold),
#             'precision_present': 0.30,
#             'recall_present': 0.70,
#             'f1_present': 0.42,
#             'accuracy': 0.61
#         }
#     }
# }

# print(f"\nInput: Mel Spectrograms ({N_MELS} mels x {int(SAMPLE_RATE * DURATION / HOP_LENGTH)} time steps)")
# print(f"Frequency range: 20-1000 Hz")
# print(f"Training: {results['train_samples']} samples (oversampled)")
# print(f"Testing: {results['test_samples']} samples")

# import json
# with open('model2_results.json', 'w') as f:
#     json.dump(results, f, indent=2)
# print("\nSaved: model2_results.json")

# fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# axes[0].hist(y_proba_positive[y_test == 0], bins=50, alpha=0.7, label='Absent', color='blue')
# axes[0].hist(y_proba_positive[y_test == 1], bins=50, alpha=0.7, label='Present', color='red')
# axes[0].axvline(x=optimal_threshold, color='green', linestyle='--', label=f'F1 thresh={optimal_threshold:.3f}')
# axes[0].axvline(x=recall_threshold, color='orange', linestyle='--', label=f'Recall thresh={recall_threshold:.3f}')
# axes[0].set_xlabel('Predicted Probability (Present)')
# axes[0].set_ylabel('Count')
# axes[0].set_title('Probability Distribution by True Class')
# axes[0].legend()

# axes[1].plot(recall_curve, precision_curve, 'b-', linewidth=2)
# axes[1].scatter(recall_curve[optimal_idx], precision_curve[optimal_idx],
#                 color='green', s=100, zorder=5, label=f'F1-opt')
# axes[1].scatter(recall_curve[target_idx], precision_curve[target_idx],
#                 color='orange', s=100, zorder=5, label=f'Recall-opt')
# axes[1].set_xlabel('Recall')
# axes[1].set_ylabel('Precision')
# axes[1].set_title('Precision-Recall Curve')
# axes[1].legend()
# axes[1].grid(True)

# thresholds_to_plot = [0.1, 0.2, 0.3, 0.4, 0.5]
# recalls = []
# precisions = []
# f1s = []
# for t in thresholds_to_plot:
#     y_pred_t = (y_proba_positive >= t).astype(int)
#     r = recall_score(y_test, y_pred_t, pos_label=1, zero_division=0)
#     p = precision_score(y_test, y_pred_t, pos_label=1, zero_division=0)
#     f = f1_score(y_test, y_pred_t, pos_label=1, zero_division=0)
#     recalls.append(r)
#     precisions.append(p)
#     f1s.append(f)

# x = np.arange(len(thresholds_to_plot))
# width = 0.25
# axes[2].bar(x - width, recalls, width, label='Recall', color='blue')
# axes[2].bar(x, precisions, width, label='Precision', color='green')
# axes[2].bar(x + width, f1s, width, label='F1', color='red')
# axes[2].set_xlabel('Threshold')
# axes[2].set_ylabel('Score')
# axes[2].set_title('Metrics at Different Thresholds')
# axes[2].set_xticks(x)
# axes[2].set_xticklabels(thresholds_to_plot)
# axes[2].legend()
# axes[2].grid(True, axis='y')

# plt.tight_layout()
# plt.savefig('m2/model2_analysis.png', dpi=150)
# plt.show()